# YOLO v8 Implementation

## Install packages

Note: If running locally, Install ultralytics and opencv-python packages in your conda virtual environment: `pip install ultralytics opencv-python `

In [ ]:
try:
    import google.colab
    print("Running on Google Colab")
    !pip install -q ultralytics opencv-python
except ModuleNotFoundError:
    print("Running locally")

from ultralytics import YOLO
from IPython.display import Image, display
import os
import matplotlib.pyplot as plt

print("Setup complete!")

## Using ```yolov8n.pt``` (Nano) model for object detection

### display the image we'll be running YOLO on

In [ ]:
from ultralytics import YOLO
import os
from IPython.display import Image, display

# 1. Load the pre-trained YOLOv8 Nano model
model = YOLO('yolov8n.pt') # The 'n' stands for Nano, which is the fastest and most lightweight version

# 2. Define our image source
# Ultralytics is smart enough to download and process an image directly from a URL
image_source = 'https://ultralytics.com/images/bus.jpg'

# Let's display 'bus.jpg' to see the image we are about to pass to our YOLO algorithm
print("Here is our input image:")
display(Image(url=image_source))

### Get output

In [ ]:
# 3. Run inference (prediction)
print("Running YOLOv8 inference...")

# 'save=True' tells YOLO to save a copy of the image with bounding boxes drawn on it
# 'conf=0.25' tells the model to ignore any predictions it is less than 25% confident about
results = model.predict(source=image_source, conf=0.25)              
results

### Understanding the Results Object

After exploring `results`, we can understand that it is a list of prediction objects, where each prediction object corresponds to one input image.

Since YOLOv8 was trained on the **COCO dataset**, the model knows **80 object classes** (such as person, car, bus, dog, etc.). However, `results` does **not** contain information for all 80 classes. It only stores predictions for the objects that were actually detected in the input image.

So, let's explore what was predicted for our target image.

`results[0].boxes` contains the final filtered detections generated after post-processing. Each detected object includes:

- Bounding box coordinates  
- Confidence score  
- Class ID (mapped to a class label)

In [ ]:
# Since we only passed one image to the model, we look at the first result: results[0]
boxes = results[0].boxes

# Let's look at the raw PyTorch tensors the model generated for all detected objects!

print("1. CLASS IDs (Which of the 80 COCO objects did it find?):")
print(boxes.cls)
print("*" * 60)

print("2. CONFIDENCE SCORES (How sure is the model, from 0.0 to 1.0?):")
print(boxes.conf)
print("*" * 60)

print("3. BOUNDING BOX COORDINATES (Center X, Center Y, Width, Height):")
print(boxes.xywh)

### Loop over each object in the input image & explore the predictions

In [ ]:
print(f"YOLO found {len(boxes)} objects in the image.\n")

# Let's loop through every single detected object and make the data readable
for box in boxes:

    # 1. Class ID & Name (What is it?)
    # .item() pulls the single number out of the complex PyTorch tensor
    class_id = int(box.cls[0].item())
    class_name = results[0].names[class_id] # Look up the actual word using the ID

    # 2. Confidence Score (How sure is the model?)
    conf = box.conf[0].item()

    # 3. Bounding Box Coordinates: Center X, Center Y, Width, Height
    # .tolist() converts the PyTorch array into a standard Python list
    coords = box.xywh[0].tolist()

    # Format the coordinates to 2 decimal places for cleaner printing
    formatted_coords = [round(num, 2) for num in coords]

    # Print the decoded results!
    print(f"Object: {class_name.upper()}")
    print(f"Confidence: {conf * 100:.1f}%")
    print(f"Coordinates (x, y, w, h): {formatted_coords}")
    print("-" * 30)

In [ ]:
# To display the YOLO result
plt.figure(figsize=(10, 7))
plt.imshow(results[0].plot()[:, :, ::-1])
plt.axis("off")
plt.show()